<a id="top"></a>
# PanSTARRS "20 queries" using MAST's TAP service: Filtering


******

## Overview

This notebook is part of a series demonstrating how to address a set of scientific questions using SQL-like Astronomical Data Query Language (ADQL) queries via a Virtual Observatory standard Table Access Protocol (TAP) service at MAST. 

This series aims to be an introduction to how complex queries can be executed using TAP (which might otherwise not be possible to specify with `astroquery`), and to be a resource for how to access MAST databases after the MAST CASJobs service is retired.

These queries are drawn from the 20 queries for SDSS as presented by [Gray, Szalay, et al. (2002)](https://arxiv.org/abs/cs/0202014), adapted for the PanSTARRS PS1 database.

This notebook presents the subset of queries which are possible through **filtering on column(s) using pre-specified criteria**, including bitmask filtering for data quality cuts.


## Learning Goals
By the end of this tutorial, you will:

- Understand how to design and perform filtering queries (with joins) using TAP services, by leveraging multiple ADQL "WHERE" constraints.


****
### Table of Contents

* [Introduction](#introduction)
* [Imports](#imports)
* [Connect to TAP service](#connect-to-tap-service)
* [Obtaining information about PanSTARRS catalogs](#obtaining-information-about-panstarrs-catalogs)
* [Q1: TB completed]()
* [Q2: Find all galaxies with blue surface brightness between 23 and 25 magnitude per square second, and super galactic latitude (sgb) between (-10$^{\circ}$, 10$^{\circ}$), and declination less than zero.](#q2)
  * [Constructing the query](#q2:-constructing-the-query)
  * [Inspecting & visualizing the results](#q2:-inspecting-&-visualizing-the-results)
* [Q16: TB completed]()
* [Conclusions](#conclusions)
* [Additional Resources](#additional-resources)
* [Citations](#citations)
* [About This Notebook](#about-this-notebook)

*******
## Introduction

Welcome! This notebook shows how to query MAST's PanSTARRS data using Astronomical Data Query Language (ADQL). We address three scientific questions by combining filtering constraints (including filtering on bitmasks for data quality cuts) using ADQL "WHERE" clauses. As you'll see in our examples, MAST uses a standard Table Access Protocol (TAP) to handle these queries.

These queries are drawn from (or closely modeled on) the "20 queries for SDSS" presented by [Gray, Szalay, et al. (2002)](https://arxiv.org/abs/cs/0202014). Taken as a whole, this collection provides worked examples on how to leverage relational database capabilities to answer scientific questions, with the aim of providing concrete starting points and references for designing queries for other research applications, both for PanSTARRS and other large data volume missions (including Roman).

<div class="alert alert-block alert-info">
<b>Note:</b> ADQL/SQL comments follow "--". Comments are used throughout the queries to explain the purpose of specific clauses.
</div>

## Imports
This tutorial makes use of the following libraries: 
- [*numpy*](https://numpy.org/) for numerical calculations
- [*pyvo*](https://pyvo.readthedocs.io) for querying the MAST catalogs via TAP
- [*matplotlib.pyplot*](https://matplotlib.org/stable/api/pyplot_summary.html#module-matplotlib.pyplot) for plotting data
- *time*, *datetime* to determine query duration
- *requests*, *warnings*, *BytesIO*, *PIL*, [*astropy.table Table*](https://docs.astropy.org/en/stable/table/index.html) to fetch & display cutout images of selected objects

In [ ]:
import numpy as np
import pyvo as vo
import matplotlib.pyplot as plt
import datetime
import time

import requests
import warnings
from io import BytesIO
from PIL import Image
from astropy.table import Table

--------
## Connect to TAP service

For all queries, we will be connecting to the PanSTARRS (PS1) Data release 2 (DR2) catalog. Specifically, we will use the `ps1_dr2` catalogs available from the new [postgres-backed TAP service](https://mast.stsci.edu/vo-tap/api/v0.1/mast_catalogs/), which offers improved performance relative to the legacy database (by factors of 100 or greater, in many cases).

See the [PS1 documentation](link_to_migration_guide_here) for information about the tables available with this new TAP service.
    
<div class="alert alert-warning" style="color:red; background-color:#ffc5c5; border-color:red;">
<b>FIX LINK TO MIGRATION GUIDE ABOVE</b>
</div>

We begin by connecting to the MAST PanSTARRS DR2 TAP service.

_(Note: See https://mast.stsci.edu/vo-tap/ for a full list of MAST TAP services.)_

In [ ]:
TAP_service = vo.dal.TAPService(
    "https://mast.stsci.edu/vo-tap/api/v0.1/mast_catalogs"
)

Using the pyvo `describe()` method, we can access an overview of the methods, capabilities, and maximum result set size for this service. 

In [ ]:
TAP_service.describe()

As expected, this service supports ADQL. You can also see the maximum result size is 100,000 rows.

## Obtaining information about PanSTARRS catalogs

In addition to the [PanSTARRS DR2 catalogs documentation](https://outerspace.stsci.edu/spaces/PANSTARRS/pages/298812351/PS1+Source+extraction+and+catalogs), 
we can search and explore information about the PS1 DR2 tables using **MAST's [Catalog Schema Browser](https://mast.stsci.edu/schema_browser)**. This schema browser provides searchable listings of column names, units, and descriptions for the PS1 DR2 tables available via this TAP service. 

We draw on this documentation and schema browser to identify the relevant tables & columns needed to address the queries below.


<div class="alert alert-warning" style="color:red; background-color:#ffc5c5; border-color:red;">
<i><b>**IF POSSIBLE:**</b></i>
<i>Integrete the schema browser.</i>
</div>

*********

## Q2

Following [Gray et al. (2002)](https://arxiv.org/abs/cs/0202014), Query 2 poses: 


> **Find all galaxies with blue surface brightness between 23 and 25 magnitude per square second, and super galactic latitude (sgb) between (-10$^{\circ}$, 10$^{\circ}$), and declination less than zero.**


### Q2: Constructing the query

For this query, we will determine surface brightnesses in the `g` band (the bluest PanSTARRS filter) using the Kron magnitude from the [`ps1_dr2.forced_mean_object` table](link_to_migration_guide).  Object positions (necessary for the RA coordinate restriction) are also available in this table (with the updated PS1 DR2 TAP service). 

    
<div class="alert alert-warning" style="color:red; background-color:#ffc5c5; border-color:red;">
Link to migration guide in the above sentence
</div>

It is necessary to join to the `ps1_dr2.stack_object` table to obtain the Kron radii, which are used in the surface brightness calculation.

For simplicity, we will also use `ramean` as a proxy instead of using the super-galactic coordinate. (Note that Gray et al. do the same.)

First, we will use

```((fmo.gfkronmag > 0) AND (fmo.gfpsfmag - fmo.gfkronmag > 0.05))```
     
to select for galaxies (excluding point sources).

Next, we will include constraints to enforce:

- the RA range
- the Dec limit
- the blue surface brightness range

We additionally will add a constraint on the `stack_object.primarydetection` key to remove duplicate entries, as `stack_object` contains duplicates of the same object measured in overlapping regions in the stack images.

Finally, to ensure we limit the query to less than the maximum number of entries for a single TAP query, we will further restrict the spatial coverage of this query to -6$^{\circ}$$\lesssim$ decmean $\lesssim$-4 $^{\circ}$.  Here we specify this by restricting the PanSTARRS `objid`s to be between 100800000000000000 and 103200000000000000 (as the first 5 digits of [PanSTARRS object identifiers](https://outerspace.stsci.edu/spaces/PanSTARRS/pages/298812384/PS1+Object+Identifiers) are determined from `floor((decl+90)/0.00833333)`).

We begin by translating the selection declination limit range into a range of `objid`s for our query constraint, following the PS1 object identifiers schema.

In [ ]:
# Determine the objid range corresponding to the selected declination range.
declstart = -6
declend = -4
objidstart = int(np.floor((declstart+90)/0.00833333))
objidend = int(np.floor((declend+90)/0.00833333))

We then construct our query by combining the constraints discussed above.  Our output is restricted to a select subset of columns and the derived $g$-band surface brightness.

In [ ]:
adql_query = f"""
SELECT fmo.objid, fmo.gfkronmag,
   fmo.gfkronmag + 2.5*log10(PI()*POWER(so.gkronrad,2)) AS gsurfmag, 
   fmo.ramean, fmo.decmean
FROM ps1_dr2.forced_mean_object as fmo
JOIN ps1_dr2.stack_object AS so ON 
    fmo.objid=so.objid
WHERE ((fmo.gfkronmag > 0) AND
       (fmo.gfpsfmag - fmo.gfkronmag > 0.05)) -- galaxies
AND fmo.ramean BETWEEN 170 AND 190            -- ra substitute for super-gal coords
AND fmo.decmean < 0
AND fmo.gfkronmag + 2.5*log10(PI()*POWER(so.gkronrad,2))
       BETWEEN 23 AND 25                      -- mag per sq arcsec
AND so.primarydetection = 1                   -- primary detection in stack_object
AND fmo.objid >= {objidstart}0000000000000            -- select decl >=-30, <-28
AND fmo.objid < {objidend}0000000000000       
"""

Next, we submit this query to the TAP service.

In [ ]:
start = time.time()
job = TAP_service.run_async(adql_query)
end = time.time()
print(f"Elapsed time: {str(datetime.timedelta(seconds=end-start))}")

This query takes about 40 seconds and returns 77,159 rows. For the entire declination range chunked by objid — 15 total chunks, from PanSTARRS' southern coverage limit at Decl=-30 to Decl=0 — this would thus take ~15 minutes (over all chunks) and return ~1.03 million rows.

### Q2: Inspecting & visualizing the results

To inspect our results, we can use the `.to_table()` method for easy viewing.

In [ ]:
TAP_results = job.to_table()
TAP_results

To visualize our results, we show a scatter plot of the selected galaxies' positions (RA/Dec), colored by their $g$-band surface brightness.

In [ ]:
# Visualize selected galaxies with a scatter plot, by blue surface brightness:
f, ax = plt.subplots(figsize=(14, 7), layout="compressed")
pts = ax.scatter(
    TAP_results['ramean'], TAP_results['decmean'],
    s=0.4, lw=0,
    c=TAP_results['gsurfmag'],
    cmap='viridis',
)
ax.set_xlim(ax.get_xlim()[::-1]) # Invert RA axis
ax.set_xlabel("RA")
ax.set_ylabel("Dec")
ax.set_aspect(1.0)          
cbar = f.colorbar(pts, aspect=5, pad=0.01)
cbar.set_label("Blue (g-band) SB")
cbar.ax.invert_yaxis()  # Invert color bar axis: surface brightness

We see our sample covers the full query spatial area, with some regions of clustering with multiple galaxies with brighter g-band surface brightnesses (e.g., at approximately (173.9, -5.8)), as we'd expect for the large scale structure of the universe.

<b style="color:red">(However, we note that some of the sources are likely spurious and caused by data artifacts, as our query does not incorporate quality cuts.)</b>

------

## Conclusions

The queries above demonstrate the speed of MAST's new, more performant PS1 TAP service.  These specific queries are up to 8 times faster than is possible with MAST's CASJobs service. (Depending on the set of constraints specified, this service can have even greater speedups.)

See the full MAST [PanSTARRS tutorial list](../../panstarrs.md) for more tutorials demonstrating how to access PanSTARRS catalogs and other select "20 queries" examples.

----------

## Additional Resources

### Table Access Protocol

- IVOA standard for RESTful web service access to tabular data
- http://www.ivoa.net/documents/TAP/

### PanSTARRS 1 DR 2

- https://outerspace.stsci.edu/display/PanSTARRS/

### Astronomical Query Data Language (2.0)

- IVOA standard for querying astronomical data in tabular format, with geometric search support
- http://www.ivoa.net/documents/latest/ADQL.html

### PyVO

- an affiliated package for [astropy](https://www.astropy.org/)
- find and retrieve astronomical data available from archives that support standard IVOA virtual observatory service protocols.
- https://pyvo.readthedocs.io/en/latest/index.html


### Full list of MAST/TAP services
- A full list of available MAST TAP services can be found at:
- https://mast.stsci.edu/vo-tap


## Citations
If you use `astropy` for published research, please cite the
authors. Follow these links for more information about citing `astropy`:

* [Citing `astropy`](https://www.astropy.org/acknowledging.html)

If you use PanSTARRS data accessed through MAST for published research, 
please include the following acknowledgements, found at the following links:

* [Acknowledging PanSTARRS](https://archive.stsci.edu/publishing/mission-acknowledgements#section-895d38a0-86b3-4143-b521-6cc3312701f9)
* [Acknowledging MAST](https://archive.stsci.edu/gsc/mast_data_use.html)


## About this Notebook

**Authors**  Rick White, Sedona Price<br>
**Keywords:** Tutorial, TAP, pyvo, ADQL, PanSTARRS <br>
**Last Updated:** August 2026
***
[Top of Page](#top)
<img style="float: right;" src="https://raw.githubusercontent.com/spacetelescope/style-guides/master/guides/images/stsci-logo.png" alt="Space Telescope Logo" width="200px"/> 